[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_Colorize.ipynb)


# AI着色デモ（Image Colorization）

顔写真をいったん **モノクロ** にし，AI（[DDColor](https://github.com/piddnad/DDColor)）で色を復元するデモです．  
**元画像 / モノクロ / 着色結果** を並べて比べられます．

**実行環境**: Google Colab（ランタイム → GPU: **T4** 推奨）

## セルの進め方
1. **設定**（Webカメラの左右反転・モデル名・入力サイズなど）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル画像のダウンロード**
4. **Gradio の起動**

> API キーは **不要** です（推論はすべて Colab 内で完結）．  
> Hugging Face のユーザー認証も **不要** です（公開モデルを自動ダウンロード）．  
> 初回はモデルのダウンロードに数十秒〜数分かかることがあります．  
> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから設定セルとセル3以降を再実行してください．


## 0. 設定

- カメラ映像が左右反転して見える場合は，次のセルの `MIRROR_WEBCAM` を切り替えてください（`True` = ミラー，`False` = 反転なし）．
- T4 では既定の `ddcolor_modelscope` を推奨します．軽い `ddcolor_paper_tiny` も選べます．
- 変更後は **Gradio 起動セル**を再実行してください．


In [ ]:
# Webカメラの左右反転（ミラー表示）
# True  : 左右反転する（Gradio のデフォルトに近い自撮り表示）
# False : 左右反転しない
MIRROR_WEBCAM = True

# DDColor の Hugging Face モデル名（piddnad/ 以下）
#   ddcolor_modelscope  : 画質重視（推奨）
#   ddcolor_artistic    : アーティスティック寄り
#   ddcolor_paper_tiny  : 軽量・高速
#   ddcolor_paper       : 論文再現用
MODEL_NAME = "ddcolor_modelscope"

# モデル入力の正方形解像度（512 が品質と速度のバランスが良い）
INPUT_SIZE = 512

# 推論前に長辺をこのピクセル以下へ縮小（VRAM・速度のバランス）
MAX_IMAGE_SIDE = 1280

print(f"MIRROR_WEBCAM = {MIRROR_WEBCAM}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"INPUT_SIZE = {INPUT_SIZE}")
print(f"MAX_IMAGE_SIDE = {MAX_IMAGE_SIDE}")


## 1. ライブラリのインストール


In [ ]:
# DDColor 本体をクローン（アーキテクチャと推論パイプライン）
# torch / opencv / Pillow / gradio / huggingface_hub / timm は Colab 標準を利用
from pathlib import Path

DDCOLOR_DIR = Path("DDColor")
if not DDCOLOR_DIR.exists():
    !git clone --depth 1 https://github.com/piddnad/DDColor.git
else:
    print(f"既に存在します: {DDCOLOR_DIR.resolve()}")

!pip install -q -U "tqdm" "huggingface_hub"
print("インストール完了")


## 2. ライブラリの読み込み，変数のインスタンス化

サンプル顔写真（カラー）をインターネットからダウンロードし，DDColor モデルを準備します．  
着色時は入力を意図的にモノクロへ変換してから AI に渡します．


In [ ]:
from __future__ import annotations

import sys
import urllib.request
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from huggingface_hub import PyTorchModelHubMixin
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

# ------------------------------------------------------------
# DDColor リポジトリを import パスへ追加
# ------------------------------------------------------------
DDCOLOR_DIR = Path("DDColor").resolve()
if not DDCOLOR_DIR.exists():
    raise FileNotFoundError(
        "DDColor が見つかりません．セル1（インストール）を先に実行してください．"
    )
if str(DDCOLOR_DIR) not in sys.path:
    sys.path.insert(0, str(DDCOLOR_DIR))

from ddcolor import DDColor, ColorizationPipeline  # noqa: E402

# ------------------------------------------------------------
# 定数・サンプル画像 URL
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_colorize")
FONT_DIR = Path("fonts")
FONT_PATH = FONT_DIR / "NotoSansJP-VF.ttf"
FONT_URL = (
    "https://raw.githubusercontent.com/googlefonts/noto-cjk/main/"
    "Sans/Variable/TTF/Subset/NotoSansJP-VF.ttf"
)

MODEL_INFO: dict[str, str] = {
    "ddcolor_modelscope": "画質重視（推奨）",
    "ddcolor_artistic": "アーティスティック寄り",
    "ddcolor_paper_tiny": "軽量・高速",
    "ddcolor_paper": "論文再現用",
}

# 公開のカラー顔写真（Wikimedia / Pexels）．日本人・アジア系を含む．
# 着色結果と「本物の色」を比べられるよう，カラー画像を用意する．
SAMPLE_IMAGE_SOURCES: list[tuple[str, str, str]] = [
    (
        "happy_japanese_woman.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/b/b0/Smiling_Japanese_Woman.jpg",
        "笑顔（日本人）",
    ),
    (
        "happy_japanese_smile.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/d/d1/Smiling_Ai_Hongo_%282024%2902.jpg",
        "スマイル（日本人）",
    ),
    (
        "neutral_japanese.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/9/90/Geisha_face_%285025641801%29.jpg",
        "ポートレート（日本人）",
    ),
    (
        "asian_portrait.jpg",
        "https://images.pexels.com/photos/1239291/pexels-photo-1239291.jpeg?auto=compress&cs=tinysrgb&w=640",
        "ポートレート（アジア系）",
    ),
    (
        "serious.jpg",
        "https://images.pexels.com/photos/2379004/pexels-photo-2379004.jpeg?auto=compress&cs=tinysrgb&w=640",
        "真剣な表情",
    ),
]

USER_AGENT = (
    "Mozilla/5.0 (compatible; OpenCampusDemo/1.0; "
    "+https://github.com/yryo1005/OpenCampus_Demo)"
)


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（時間がかかります）．")
    return "cpu"


def download_file(url: str, save_path: Path, max_side: int = 1280) -> Path:
    """URL から画像をダウンロードし，必要なら長辺を縮小して保存する．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス
        max_side (int): 長辺の上限ピクセル（既定 1280）

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        raw = response.read()
    arr = np.frombuffer(raw, dtype=np.uint8)
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"画像のデコードに失敗しました: {url}")
    h, w = bgr.shape[:2]
    long_side = max(h, w)
    if long_side > max_side:
        scale = max_side / float(long_side)
        bgr = cv2.resize(
            bgr,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    ok, encoded = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError(f"画像のエンコードに失敗しました: {save_path}")
    save_path.write_bytes(encoded.tobytes())
    return save_path


def download_font(url: str, save_path: Path) -> Path:
    """日本語表示用フォントをダウンロードする（既存ならスキップ）．

    Args:
        url (str): フォントの URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したフォントのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=120) as response:
        save_path.write_bytes(response.read())
    return save_path


def prepare_sample_images(
    sources: list[tuple[str, str, str]],
    sample_dir: Path,
) -> list[tuple[str, Path]]:
    """サンプル顔写真をダウンロードし，ラベルとパスの一覧を返す．

    Args:
        sources (list[tuple[str, str, str]]): (ファイル名, URL, 表示ラベル) のリスト
        sample_dir (Path): 保存先ディレクトリ

    Returns:
        list[tuple[str, Path]]: (表示ラベル, ローカルパス) のリスト
    """
    prepared: list[tuple[str, Path]] = []
    for filename, url, label in tqdm(sources, desc="サンプル画像DL", leave=False):
        path = download_file(url, sample_dir / filename)
        print(f"  {label}: {path} ({path.stat().st_size} bytes)")
        prepared.append((label, path))
    return prepared


class DDColorHF(DDColor, PyTorchModelHubMixin):
    """Hugging Face Hub 経由で重みを読み込む DDColor ラッパ．"""

    def __init__(self, config=None, **kwargs):
        if isinstance(config, dict):
            kwargs = {**config, **kwargs}
        super().__init__(**kwargs)


def load_colorizer(model_name: str, input_size: int, device: str) -> ColorizationPipeline:
    """DDColor を Hugging Face から読み込み，着色パイプラインを返す．

    Args:
        model_name (str): HF リポジトリ名（例: ddcolor_modelscope）
        input_size (int): モデル入力の一辺（ピクセル）
        device (str): "cuda" または "cpu"

    Returns:
        ColorizationPipeline: BGR uint8 → BGR uint8 の着色器
    """
    repo_id = model_name if "/" in model_name else f"piddnad/{model_name}"
    print(f"モデル読込中: {repo_id}")
    model = DDColorHF.from_pretrained(repo_id)
    model = model.to(device)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"読込完了: {n_params:.1f}M parameters, device={device}")
    return ColorizationPipeline(
        model,
        input_size=input_size,
        device=torch.device(device),
    )


def to_rgb_uint8(image) -> np.ndarray | None:
    """Gradio / PIL / ndarray 入力を RGB uint8 (H, W, 3) に揃える．

    Args:
        image: Gradio Image の入力（None / PIL.Image / np.ndarray）

    Returns:
        np.ndarray | None: RGB 画像．入力が無い場合は None
    """
    if image is None:
        return None
    if isinstance(image, Image.Image):
        return np.asarray(image.convert("RGB"))
    arr = np.asarray(image)
    if arr.ndim == 2:
        return cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    if arr.shape[2] == 4:
        return arr[:, :, :3].astype(np.uint8)
    return arr.astype(np.uint8)


def resize_max_side(rgb: np.ndarray, max_side: int) -> np.ndarray:
    """長辺が max_side を超える場合に縮小する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        max_side (int): 長辺の上限

    Returns:
        np.ndarray: 必要なら縮小した RGB 画像
    """
    h, w = rgb.shape[:2]
    long_side = max(h, w)
    if long_side <= max_side:
        return rgb
    scale = max_side / float(long_side)
    return cv2.resize(
        rgb,
        (int(w * scale), int(h * scale)),
        interpolation=cv2.INTER_AREA,
    )


def to_grayscale_rgb(rgb: np.ndarray) -> np.ndarray:
    """カラー画像を意図的にモノクロ（3ch RGB）へ変換する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)，dtype uint8

    Returns:
        np.ndarray: モノクロ化した RGB 画像，形状 (H, W, 3)
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)


def colorize_rgb(gray_rgb: np.ndarray, colorizer: ColorizationPipeline) -> np.ndarray:
    """モノクロ RGB 画像を DDColor で着色し，RGB で返す．

    Args:
        gray_rgb (np.ndarray): モノクロ RGB，形状 (H, W, 3)，dtype uint8
        colorizer (ColorizationPipeline): DDColor パイプライン

    Returns:
        np.ndarray: 着色結果 RGB，形状 (H, W, 3)，dtype uint8
    """
    bgr = cv2.cvtColor(gray_rgb, cv2.COLOR_RGB2BGR)
    out_bgr = colorizer.process(bgr)
    return cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)


def _load_jp_font(size: int) -> ImageFont.ImageFont:
    """日本語ラベル用フォントを返す．

    Args:
        size (int): フォントサイズ（ピクセル）

    Returns:
        ImageFont.ImageFont: 利用可能なフォント
    """
    if FONT_PATH.exists():
        return ImageFont.truetype(str(FONT_PATH), size=size)
    return ImageFont.load_default()


def make_triple_compare(
    original: np.ndarray,
    grayscale: np.ndarray,
    colorized: np.ndarray,
    labels: tuple[str, str, str] = ("元画像", "モノクロ", "AI着色"),
) -> np.ndarray:
    """元画像・モノクロ・着色結果を横並びにし，ラベルを付ける．

    Args:
        original (np.ndarray): 元画像 RGB (H, W, 3)
        grayscale (np.ndarray): モノクロ RGB (H, W, 3)
        colorized (np.ndarray): 着色 RGB (H, W, 3)
        labels (tuple[str, str, str]): 各パネルの日本語ラベル

    Returns:
        np.ndarray: 横並び比較画像 RGB，形状 (H+帯, 3W, 3)
    """
    h = min(original.shape[0], grayscale.shape[0], colorized.shape[0])
    w = min(original.shape[1], grayscale.shape[1], colorized.shape[1])

    def _fit(img: np.ndarray) -> np.ndarray:
        if img.shape[0] == h and img.shape[1] == w:
            return img
        return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)

    panels = [_fit(original), _fit(grayscale), _fit(colorized)]
    strip = np.concatenate(panels, axis=1)

    bar_h = max(36, h // 16)
    canvas = Image.new("RGB", (strip.shape[1], h + bar_h), (32, 32, 32))
    canvas.paste(Image.fromarray(strip), (0, bar_h))
    draw = ImageDraw.Draw(canvas)
    font = _load_jp_font(size=max(16, bar_h // 2))
    for i, label in enumerate(labels):
        x = i * w + 8
        draw.text((x, 6), label, fill=(255, 255, 255), font=font)
    return np.asarray(canvas)


def colorize_face(
    image,
    mirror: bool = False,
) -> tuple[np.ndarray | None, np.ndarray | None, np.ndarray | None, np.ndarray | None, str]:
    """Gradio 用：入力をモノクロ化し，DDColor で着色して比較画像を返す．

    Args:
        image: Gradio Image 入力（None / PIL / ndarray）
        mirror (bool): True のとき水平フリップしてから処理する

    Returns:
        tuple: (元画像, モノクロ, 着色, 三点比較, 説明文)
            各画像は RGB uint8．入力が無い場合は画像が None
    """
    rgb = to_rgb_uint8(image)
    if rgb is None:
        return None, None, None, None, "画像を入力してください（カメラ／アップロード／サンプル）．"

    with tqdm(total=4, desc="着色", leave=False) as pbar:
        if mirror:
            rgb = cv2.flip(rgb, 1)
        rgb = resize_max_side(rgb, MAX_IMAGE_SIDE)
        pbar.update(1)

        gray_rgb = to_grayscale_rgb(rgb)
        pbar.update(1)

        colorized = colorize_rgb(gray_rgb, colorizer)
        pbar.update(1)

        compare = make_triple_compare(rgb, gray_rgb, colorized)
        pbar.update(1)

    model_ja = MODEL_INFO.get(MODEL_NAME, MODEL_NAME)
    text = (
        f"【モデル】DDColor / {MODEL_NAME}（{model_ja}）\n"
        f"【入力解像度】{INPUT_SIZE}×{INPUT_SIZE}\n"
        f"【処理サイズ】{rgb.shape[1]}×{rgb.shape[0]}\n"
        f"【左右反転】{'あり' if mirror else 'なし'}\n\n"
        "カラー写真 → 意図的にモノクロ化 → AIが色を推定，という流れです．\n"
        "元の色と AI の色が違うことがあります（それが面白いポイントです）．"
    )
    return rgb, gray_rgb, colorized, compare, text


def build_demo(
    sample_items: list[tuple[str, Path]],
    mirror_webcam: bool,
) -> gr.Blocks:
    """Gradio UI を構築する．

    Args:
        sample_items (list[tuple[str, Path]]): (表示ラベル, 画像パス)
        mirror_webcam (bool): カメラ入力を左右反転するか

    Returns:
        gr.Blocks: Gradio デモ
    """
    example_paths = [str(path) for _, path in sample_items]

    with gr.Blocks(title="AI着色デモ") as demo:
        gr.Markdown(
            """
            # AI着色デモ
            顔写真をいったん **モノクロ** にし，AI（DDColor）で色を復元します．  
            **カメラ**で撮影するか，下の**サンプル画像**をクリックして試せます．
            """
        )
        with gr.Row():
            with gr.Column():
                image_in = gr.Image(
                    label="顔写真（カメラ / アップロード）",
                    type="numpy",
                    sources=["webcam", "upload"],
                    webcam_options=gr.WebcamOptions(mirror=mirror_webcam),
                )
                mirror_flag = gr.Checkbox(
                    label="入力画像を左右反転して着色する",
                    value=False,
                    info="アップロード画像の向きが逆のときだけオンにしてください（カメラは上のミラー設定を利用）",
                )
                run_btn = gr.Button("AI着色", variant="primary")
            with gr.Column():
                compare_out = gr.Image(label="元画像 / モノクロ / AI着色", type="numpy")
                text_out = gr.Textbox(label="説明", lines=8)

        with gr.Row():
            original_out = gr.Image(label="元画像", type="numpy")
            gray_out = gr.Image(label="モノクロ画像", type="numpy")
            color_out = gr.Image(label="着色画像", type="numpy")

        gr.Examples(
            examples=example_paths,
            inputs=[image_in],
            label="サンプル画像（クリックで入力）",
            examples_per_page=8,
        )

        outputs = [original_out, gray_out, color_out, compare_out, text_out]
        run_btn.click(
            fn=colorize_face,
            inputs=[image_in, mirror_flag],
            outputs=outputs,
        )
        image_in.change(
            fn=colorize_face,
            inputs=[image_in, mirror_flag],
            outputs=outputs,
        )
    return demo


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
device = resolve_device()
download_font(FONT_URL, FONT_PATH)
print(f"フォント: {FONT_PATH} ({FONT_PATH.stat().st_size} bytes)")
sample_items = prepare_sample_images(SAMPLE_IMAGE_SOURCES, SAMPLE_DIR)
colorizer = load_colorizer(MODEL_NAME, INPUT_SIZE, device)
print("初期化完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行


In [ ]:
demo = build_demo(sample_items, mirror_webcam=MIRROR_WEBCAM)
demo.launch(share=True, debug=False)
